<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat%2Fp3-soft-router/notebooks/p3_soft_router_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Thực Nghiệm P3: Soft Top-K Prompt Routing & Prompt Composition (TAPTA Minimal)
### Protocol v1 Track A (FewRel 8 tasks × 10 relations, strict 5-shot, Rehearsal-Free $M=0$, Seed 2021)

Notebook này thực thi phương pháp đề xuất **P3 (Soft Top-K Prompt Composition Router)** trên môi trường **Google Colab GPU (CUDA)** mà **hoàn toàn không có Oracle Task ID lúc kiểm thử**:
- **Khắc phục nhược điểm của P2a:** Thay vì chọn cứng 1 prompt đơn lẻ ($K=1$, rủi ro méo mó đặc trưng khi chọn sai tác vụ), P3 tính trọng số mềm qua Softmax nhiệt độ $\tau=0.2$ trên Top-$K$ ($K=2$) tác vụ ứng viên và tổ hợp mồi nhắc riêng biệt cho từng mẫu:
  $$P_{\text{soft}}(x) = \sum_{k \in \text{Top-}K} w_k(x) P_k$$
- **Prompt Orthogonality Regularization:** Thêm hàm mất mát trực giao $\mathcal{L}_{\text{orth}}$ ($\lambda=0.05$) giữa prompt mới và các prompt cũ để các mồi nhắc không bị sụp đổ/trùng lặp.
- **Mốc so sánh:**
  - P0 (Shared Prompt — Cận dưới): `52.83%`
  - P2b (Learned Router): `56.04%` (Router Acc: `50.85%`)
  - P2a (Centroid Hard Router): `59.35%` (Router Acc: `64.69%`)
  - P1 (Oracle-Task Prompt — Cận trên lý tưởng): `70.14%`
  - **Mục tiêu P3:** Đạt Whole Accuracy $\ge 62.0\%$, thu hẹp $\ge 50\%$ khoảng cách tới trần Oracle P1.

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [ ]:
import torch
print('=' * 60)
print('CUDA / GPU PREFLIGHT CHECK')
print('=' * 60)
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device:     {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    raise RuntimeError('Vui lòng bật GPU trên Google Colab: Runtime -> Change runtime type -> T4 GPU!')

## 2. Clone / Cập nhật Repository `research-code-lab` (Nhánh `feat/p3-soft-router`)

In [ ]:
import os
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/p3-soft-router'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd /content/research-code-lab
!pip install -q transformers pyyaml

## 3. Chạy Thực Nghiệm P3: Soft Top-K Prompt Routing & Composition

In [ ]:
!python experiments/run_p3.py \
    --config configs/p3_soft_router_fewrel_5shot_seed2021.yaml \
    --device cuda

## 4. Kiểm tra Kết quả & Đối sánh Hiệu năng (P0 vs P2 vs P3 vs P1)

In [ ]:
import json
from pathlib import Path

results_path = Path('runs/p3_soft_router/fewrel_5shot/seed_2021/summary.json')
if results_path.exists():
    with results_path.open() as f:
        p3_summary = json.load(f)
    
    print('=' * 75)
    print('BẢNG ĐỐI SÁNH HIỆU NĂNG TỔNG HỢP (PROTOCOL V1 TRACK A FEWREL 5-SHOT)')
    print('=' * 75)
    print(f'{"Mô hình":<30} | {"Proto Acc":<10} | {"Router Acc":<12} | {"Top-K Cov":<10} | {"Gap Closed":<10}')
    print('-' * 75)
    print(f'{"P0: Shared Prompt (Sàn)":<30} | 52.83%     | N/A          | N/A        | 0.00%')
    print(f'{"P2b: Learned MLP Router":<30} | 56.04%     | 50.85%       | N/A        | 18.56%')
    print(f'{"P2a: Centroid Cosine Router":<30} | 59.35%     | 64.69%       | N/A        | 37.69%')
    
    p3_acc = p3_summary.get('final_prototype_accuracy', 0.0) * 100
    p3_topk_acc = p3_summary.get('final_topk_gated_prototype_accuracy', 0.0) * 100
    p3_r1 = p3_summary.get('final_router_top1_accuracy', 0.0) * 100
    p3_rk = p3_summary.get('final_router_topk_coverage', 0.0) * 100
    p3_gap = p3_summary.get('oracle_gap_closed_pct', 0.0)
    
    print(f'{"P3: Soft Top-K Router":<30} | {p3_acc:5.2f}%     | {p3_r1:5.2f}%       | {p3_rk:5.2f}%    | {p3_gap:5.2f}%')
    print(f'{"P3: Top-K Gated":<30} | {p3_topk_acc:5.2f}%     | -            | -          | -')
    print(f'{"P1: Oracle-Task (Trần)":<30} | 70.14%     | 100.00%      | 100.00%    | 100.00%')
    print('=' * 75)
else:
    print('Chưa tìm thấy file summary.json. Vui lòng chạy cell 3 trước.')

## 5. Đóng gói & Tải về Kết quả (Download Results Artifact)

In [ ]:
import shutil
from google.colab import files

archive_dir = 'runs/p3_soft_router/fewrel_5shot/seed_2021'
zip_name = '/content/p3_soft_router_results_seed2021'

shutil.make_archive(zip_name, 'zip', archive_dir)
print(f'Đã đóng gói thành công: {zip_name}.zip')

# Tự động tải về máy
files.download(f'{zip_name}.zip')